# XGBoost vs SPLIT vs RESPLIT vs PRAXIS Comparison

Compare four algorithms on airline passenger satisfaction dataset:
- **XGBoost**: Black-box ensemble baseline
- **SPLIT**: Glass-box single optimal tree
- **RESPLIT**: Rashomon set enumeration (command-line)
- **PRAXIS**: Proxy-based Rashomon enumeration

## Part 1: XGBoost and SPLIT Training

In [14]:
import dimex as dx
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

SELECTED_FEATURES = [
    'Online_boarding', 'Type_of_Travel_Personal Travel', 'Class_Eco',
    'Inflight_wifi_service', 'On_board_service', 'Customer_Type_disloyal Customer',
    'Inflight_entertainment', 'Checkin_service', 'Leg_room_service'
]

# Load training data (unbalanced, sample 10k with stratification)
train_data = pd.read_csv('../airline-passenger-satisfaction/train_clean_encoded.csv')

# Stratified sample: keep ~50% of each class from the 10k sample
train_data = train_data.groupby('satisfaction', group_keys=False).apply(
    lambda x: x.sample(n=5000, random_state=42)
)
print(f"Sampled 10k rows: {train_data.shape}")

# Balance via SMOTE
x_train, y_train = dx.smote(train_data[SELECTED_FEATURES], train_data['satisfaction'], train_data)
print(f"Training set shape after SMOTE: {x_train.shape}")

Sampled 10k rows: (10000, 24)


/tmp/ipykernel_3366/423747300.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_data = train_data.groupby('satisfaction', group_keys=False).apply(


TypeError: expected str, bytes or os.PathLike object, not DataFrame

In [ ]:
# Load test data
test_data = pd.read_csv('../airline-passenger-satisfaction/test_clean_encoded.csv')
x_test = test_data[SELECTED_FEATURES]
y_test = test_data['satisfaction']

results = []
SPLIT_REG = 0.005

# 1. Train XGBoost
print("Training XGBoost...")
xgb_model, xgb_size, xgb_runtime = dx.train_xgb(x_train, y_train)
xgb_pred, _ = dx.prediction_xgb(xgb_model, x_test, y_test)
xgb_loss = (xgb_pred != y_test).sum() / len(y_test)
results.append({
    'Model': 'XGBoost',
    'Accuracy': accuracy_score(y_test, xgb_pred),
    'Precision': precision_score(y_test, xgb_pred),
    'Recall': recall_score(y_test, xgb_pred),
    'F1': f1_score(y_test, xgb_pred),
    'Loss': round(xgb_loss, 6),
    'Size': f"{xgb_size['trees']} trees, {xgb_size['leaves']} leaves",
    'Runtime (s)': round(xgb_runtime, 3)
})

# 2. Train SPLIT
print("Training SPLIT...")
split_model, split_tree, split_meta = dx.train_split(x_train, y_train, lookahead=2, full_depth=6, reg=SPLIT_REG)
split_pred, _ = dx.prediction_split(split_model, x_test, y_test)
split_loss = (split_pred == y_test).sum() / len(y_test) + split_meta['leaves'] * SPLIT_REG
results.append({
    'Model': 'SPLIT',
    'Accuracy': accuracy_score(y_test, split_pred),
    'Precision': precision_score(y_test, split_pred),
    'Recall': recall_score(y_test, split_pred),
    'F1': f1_score(y_test, split_pred),
    'Loss': round(split_loss, 6),
    'Size': f"{split_meta['leaves']} leaves",
    'Runtime (s)': round(split_meta['runtime'], 3)
})

display(pd.DataFrame(results))

In [4]:
import json

with open('results1.json', 'w') as f:
    json.dump(results, f, indent=4)

## Part 2: Load RESPLIT Results (from command-line execution)

Run `python run_resplit.py` from the terminal first, then execute this cell.

In [ ]:
import json

# Load RESPLIT results from run_resplit.py output
try:
    with open('../results/resplit_metrics.json', 'r') as f:
        resplit_results = json.load(f)
    # Add average RESPLIT result to comparison
    avg_accuracy = sum(r['Accuracy'] for r in resplit_results) / len(resplit_results)
    avg_precision = sum(r['Precision'] for r in resplit_results) / len(resplit_results)
    avg_recall = sum(r['Recall'] for r in resplit_results) / len(resplit_results)
    avg_f1 = sum(r['F1'] for r in resplit_results) / len(resplit_results)
    
    results.append({
        'Model': f'RESPLIT (avg of {len(resplit_results)} trees)',
        'Accuracy': avg_accuracy,
        'Precision': avg_precision,
        'Recall': avg_recall,
        'F1': avg_f1,
        'Size': f"{len(resplit_results)} trees"
    })
    print(f"✓ Loaded RESPLIT results ({len(resplit_results)} trees)")
except FileNotFoundError:
    print("⚠ RESPLIT results not found. Run `python run_resplit.py` first.")

## Part 3: PRAXIS Training

In [ ]:
# 3. Train PRAXIS
print("Training PRAXIS...")
from praxis import PRAXIS
praxis_model = PRAXIS(lambda_reg=0.005, depth_budget=6, rashomon_mult=0.03, lookahead_k=1)
praxis_model.fit(x_train, y_train)

praxis_pred_single = praxis_model.predict(x_test)
praxis_pred_ensemble = praxis_model.predict_ensemble(x_test)

results.append({
    'Model': 'PRAXIS (Best Tree)',
    'Accuracy': accuracy_score(y_test, praxis_pred_single),
    'Precision': precision_score(y_test, praxis_pred_single),
    'Recall': recall_score(y_test, praxis_pred_single),
    'F1': f1_score(y_test, praxis_pred_single),
    'Size': f"1 of {len(praxis_model)} trees"
})

results.append({
    'Model': 'PRAXIS (Ensemble)',
    'Accuracy': accuracy_score(y_test, praxis_pred_ensemble),
    'Precision': precision_score(y_test, praxis_pred_ensemble),
    'Recall': recall_score(y_test, praxis_pred_ensemble),
    'F1': f1_score(y_test, praxis_pred_ensemble),
    'Size': f"{len(praxis_model)} trees (voting)"
})

# Final comparison table
results_df = pd.DataFrame(results)
print("\n" + "="*100)
print("FINAL COMPARISON: XGBoost vs SPLIT vs RESPLIT vs PRAXIS")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)